# Twinned FCC 3D Pipeline: Basic Walkthrough

A concise, end-to-end demonstration of the Twinned FCC pipeline, from EBSD
input through Abaqus mesh export. Each validation stage is reported as a
single pass/fail check, and misorientation-distribution agreement is
illustrated with one comparison plot; the diagnostic plots and tuning
parameters retained in the companion notebooks are omitted here for
brevity.

Related notebooks in this directory: `twinned_fcc_int0.ipynb` (validation
plots and the principal Host Allocation and Twin Generation parameters
exposed) and `twinned_fcc_adv0.ipynb` (additional tuning parameters beyond
the intermediate tier).

**Pipeline stages**: EBSD analysis, Monte Carlo base grain structure
generation, representativeness assessment and qualification, host
allocation, orientation assignment, twin generation, post-twin validation,
and visualization/export. The Voronoi and image-import base-structure
paths are not covered; Monte Carlo is the default generation method
throughout.

Parameter defaults correspond to values validated against the reference
dataset used in this notebook.

## Part A: Start (Global Configuration)

Establishes the pipeline output directory and opens the report session
(`upxo.reporting.ReportSession`) to which subsequent stages append
results. All outputs generated by this notebook, including raw exports,
Abaqus mesh files, and the report itself, are written under
`<output_dir>/TwinnedFCC/...`.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

from upxo.pxtal.twinned_simple_3d.steps import (
    steps_start, steps_ebsd_analysis_1, steps_self_repr_1, steps_ebsd_analysis_2,
    steps_base_grain_structure_mc, steps_repr_assessment_qualification,
    steps_transformations, steps_host_allocation, steps_pre_twin_validation,
    steps_orientation_assignment, steps_twin_generation, steps_post_twin_validation,
    steps_distribution_viewer, steps_subsetting, steps_visualization_export,
)

In [ ]:
RESEARCHER_NAME = ""
ORGANISATION = ""
EMAIL = ""
MATERIAL_NAME = "OFHC Copper"
PROCESSING_CONDITION = ""

report = steps_start.open_pipeline_report(
    researcher_name=RESEARCHER_NAME, organisation=ORGANISATION, email=EMAIL,
    material_name=MATERIAL_NAME, processing_condition=PROCESSING_CONDITION,
    notebook_tag='bas0')
report

## Part B: EBSD Analysis-1

Loads the reference EBSD map (subsampled by default, as grain detection on
full-resolution maps is computationally expensive), performs grain
detection, and trims a one-percent border to avoid edge artefacts.
Produces the `repgen2d` object (`rg`) used throughout the remainder of the
pipeline.

In [ ]:
CTF_FILE = r'C:\Development\EBSD datasets\UKAEA__OFHCCu\OFHC_Cu_dataset\EBSD_pre\warp_out_s2.ctf'
STRIDE_X, STRIDE_Y = 5, 5   # default subsampling

rdr = steps_ebsd_analysis_1.subsample_and_load(
    CTF_FILE, subsample=True, stride_x=STRIDE_X, stride_y=STRIDE_Y, reuse_subsampled=True)

In [ ]:
MIN_GRAIN_SIZE_DETECT = 10   # pixels
MISORI_TOL = 10.0           # degrees
rdr = steps_ebsd_analysis_1.detect_grains(rdr, min_grain_size=MIN_GRAIN_SIZE_DETECT, misori_tol=MISORI_TOL)

In [ ]:
rdr = steps_ebsd_analysis_1.crop(rdr, xstart_pct=1.0, ystart_pct=1.0, xend_pct=99.0, yend_pct=99.0)

In [ ]:
rg = steps_ebsd_analysis_1.clean_and_characterize(rdr, ctf_file=CTF_FILE)

## Part C: Self-Representativeness-1 (Omitted)

An optional self-representativeness study of the EBSD domain, omitted here
for brevity. See `twinned_fcc_int0.ipynb` or `twinned_fcc_adv0.ipynb` for
the complete analysis.

## Part D: EBSD Analysis-2

The most computationally intensive stage of the analysis. Computes the
misorientation distribution function (MDF), detects candidate
coincidence-site-lattice (CSL) peaks, segregates parent/twin grain pairs
about the Sigma3 (twin) peak, classifies the twin role of every grain,
partitions the twin volume fraction into Stage-1/Stage-2a/Stage-2b
targets, and measures twin lamella thickness. The resulting quantities
(`parent_info`, `vf_targets`, `twin_thickness`) constitute inputs to
nearly every subsequent stage.

In [ ]:
mdf, peaks = steps_ebsd_analysis_2.compute_mdf_and_peaks(rg)
selected_peaks = steps_ebsd_analysis_2.select_all_peaks(peaks)   # keep every detected peak
len(selected_peaks['indices'])

In [ ]:
csl_grains = steps_ebsd_analysis_2.segregate_csl_pairs(rg, mdf, peaks, selected_peaks)
parent_info = steps_ebsd_analysis_2.identify_parent_grains(rg, csl_grains)

In [ ]:
CSL_LABEL = steps_ebsd_analysis_2.DEFAULT_CSL_LABEL   # 'S3  (twin)'
tvf_by_label = steps_ebsd_analysis_2.compute_twin_area_fractions(rg, parent_info, csl_labels=(CSL_LABEL,))
tvf_by_label[CSL_LABEL]

In [ ]:
role_props = steps_ebsd_analysis_2.compute_grain_role_properties(rg, parent_info, selected_props=('area',))
vf_partition, vf_targets = steps_ebsd_analysis_2.compute_vf_partition(rg, parent_info, tvf_by_label, csl_label=CSL_LABEL)
vf_targets

In [ ]:
twin_thickness = steps_ebsd_analysis_2.compute_twin_thickness(rg, parent_info)
twin_thickness['mean'], twin_thickness['median']

In [ ]:
report.add_table([vf_targets], title='EBSD VF Targets', section='EBSD Analysis-2')
twin_thickness_summary = {k: v for k, v in twin_thickness.items() if not hasattr(v, '__len__') or isinstance(v, str)}
report.add_table([twin_thickness_summary], title='EBSD Twin Thickness Statistics', section='EBSD Analysis-2')

## Part E: Self-Representativeness-3 (Omitted)

Texture and pole-figure analysis for this stage remains unimplemented in
the current package; no functional code exists to wrap. Consistent with
the `twinned_simple_3d` package documentation, which advises against
placeholder implementations for production studies, no
`steps_self_repr_3.py` module or corresponding cell is provided here.

## Part F: Base Grain Structure (Monte Carlo)

Method selection is unambiguous on this path: Monte Carlo is the
base-grain-structure generation method addressed in this notebook,
distinct from the Voronoi and image-import paths, which are not covered
here. Executes a three-dimensional Potts-model Monte Carlo grain-growth
simulation, computes the labelled feature index (grain labels) for every
stored temporal slice, and applies cleaning to each slice (merging
undersized grains, removing single-voxel spikes).

In [ ]:
# NOTE: a full 50x50x50, 100-step run can take a while --
# shrink XMAX/YMAX/ZMAX/MCSTEPS for a quick smoke-test run.
pxt = steps_base_grain_structure_mc.run_mc_simulation(
    xmax=50.0, ymax=50.0, zmax=50.0, q_states=10, mcsteps=100, save_interval=5,
    mcalg='300b', consider_boltzmann=True, boltzmann_temp_factor=0.01, rng_seed=0)

In [ ]:
lfi = steps_base_grain_structure_mc.calculate_lfi(pxt)

In [ ]:
cumulative_summary, n_passes_run = steps_base_grain_structure_mc.clean_structure(pxt, min_grain_size=4, start_index=1)
n_passes_run

## Part G: Representativeness Assessment and Qualification


Ranks every stored Monte Carlo temporal slice against the EBSD
parent-grain reference (two-dimensional cross-sections compared with the
EBSD map), re-assesses candidates under the configured tolerances, and
selects the single best-matching slice as the authoritative structure for
host allocation. The physical scale factor (um/voxel) of the selected
slice is calibrated in the same step.

Ranking begins at slice index 1 rather than 0: slice 0 represents the
Monte Carlo simulation's un-annealed seed state (a grain count equal to
the voxel count, with no growth), which is not a meaningful
synthetic-structure candidate. Its disproportionately large grain count
can otherwise dominate the shortlist under certain criteria despite
representing the poorest overall match. The Base Grain Structure cleaning
step already excludes this slice (`start_index=1`) for the same reason.

In [ ]:
candidates = steps_repr_assessment_qualification.rank_slices(
    pxt, parent_info=parent_info, role_props=role_props, start=1)
len(candidates)

In [ ]:
candidates = steps_repr_assessment_qualification.reassess_candidates(pxt, candidates, role_props)

In [ ]:
rows, best_row = steps_repr_assessment_qualification.shortlist_and_select(candidates, role_props)
best_row['tslice_key'] if best_row else None

In [ ]:
scale_factor = steps_repr_assessment_qualification.apply_selected_scale_factor(pxt, best_row)
tslice_key = best_row['tslice_key']
scale_factor

In [ ]:
report.add_table([rows[0]] if rows else [{}], title='Temporal Slice Shortlist', section='Repr. assess. & qualification')

## Part H: Transformations (Omitted)

An optional non-equiaxiality rescale/stretch of the base structure,
omitted in this basic walkthrough; the identity transform (no
modification) is applied throughout. See
`twinned_fcc_int0.ipynb`/`twinned_fcc_adv0.ipynb` to enable this stage.

## Part I: Host Allocation


Allocates which grains within the base structure become twin "hosts", via
a maximal-independent-set-constrained spatial selection targeting a
specified volume fraction, then re-assesses the result through
two-dimensional cross-sections for comparability with the EBSD
two-dimensional hosting-fraction target.

`HOST_TARGET_FRACTION` is a grain-count fraction, derived from EBSD's
`twin_hosting_fraction` (the fraction of parent grains hosting a twin),
not an area or volume fraction; the achieved hosting fraction reported
below is, by contrast, a volume fraction, and the two quantities are
therefore not directly comparable. The volume captured by a given
grain-count selection depends strongly on `HOST_RANKING_VOLUME_WEIGHT` and
`HOST_POOL_B_BAND_SHAPE`: ranking purely by volume (1.0) and drawing Pool
B from the large-grain tail (`'above'`), as configured here, biases
selection toward larger grains, such that a moderate grain-count fraction
can capture a disproportionate share of total volume on a right-skewed
grain-size distribution.

In [ ]:
HOST_TARGET_FRACTION = 0.3          # EBSD 2D hosting fraction (grain-count based)
HOST_MIN_VOXELS = 4                 # grains smaller than this are never host-eligible
HOST_2D_TO_3D_SCALE_FACTOR = 1.65   # corrects 2D EBSD under-sampling of the true 3D hosting fraction
HOST_RANKING_VOLUME_WEIGHT = 1.0    # 0=rank by adjacency count, 1=rank by volume (largest first), 0.5=equal blend
HOST_MIS_FRACTION = 0.25            # fraction of hosts drawn from the non-adjacent (MIS) pool
HOST_MIS_RUNS = 10                  # number of random MIS trials; the largest is kept
HOST_SEED = 0
HOST_POOL_B_BAND_SHAPE = 'above'    # 'below'/'between'/'above' -- which size-tail Pool B draws from
HOST_POOL_B_N_STD = 1.0             # Pool B band half-width, in std devs of grain size

base = steps_host_allocation.allocate_hosts(
    pxt, tslice_key, transformed_base=None,   # Part H (Transformations) is skipped in this notebook
    target_fraction=HOST_TARGET_FRACTION, min_voxels=HOST_MIN_VOXELS,
    scale_factor=HOST_2D_TO_3D_SCALE_FACTOR, ranking_weight=HOST_RANKING_VOLUME_WEIGHT,
    mis_fraction=HOST_MIS_FRACTION, mis_runs=HOST_MIS_RUNS, seed=HOST_SEED,
    pool_b_band_shape=HOST_POOL_B_BAND_SHAPE, pool_b_n_std=HOST_POOL_B_N_STD)

In [ ]:
hosting_2d = steps_host_allocation.reassess_hosting_2d(base)
hosting_2d

In [ ]:
report.add_table([hosting_2d], title='Host Allocation 2D Re-Assessment', section='Host Allocation')

## Part J: Pre-Twin Validation


Validates the grain-size distribution of the host-allocated base structure
against the EBSD twin-merged reference (twins merged into their parent
grain, as the base structure does not yet contain twins), per axis, via
two-dimensional cross-sections and a Wasserstein-distance pass
threshold.

In [ ]:
pre_validator = steps_pre_twin_validation.validate_morphological_representativeness(
    base, rg, parent_info, n_slices=(10, 10, 10), test_axes=('x', 'y', 'z'),
    p_percentage=(60.0, 60.0, 60.0), wasserstein_threshold=0.5)
pre_validator.overall_accepted

In [ ]:
print(pre_validator.report())

## Part K: Orientation Assignment

Assigns a crystal orientation (quaternion) to every grain of the
host-allocated base structure: EBSD-derived parent-grain orientations for
host grains, and a fallback texture pool for non-host grains, with
twin-pair conflicts resolved according to the selected mode.
`conflict_free`, the simplest and fastest mode, is used by default.

In [ ]:
assigner = steps_orientation_assignment.assign_orientations(
    base, rg, parent_info, csl_label=CSL_LABEL, ori_mode='conflict_free', rng_seed=42)
assigner.n_conflicts

## Part L: Twin Generation

Introduces primary twins (at host grain-boundary centroids by default),
followed by secondary twins nucleating inward or outward from the primary
twins, targeting the EBSD twin-area fraction measured in Part D. The
resulting structure is then topologically cleaned, removing spike voxels
and splitting non-convex ("lobed") grains produced by the voxel-carving
procedure.

`vf_targets` (Part D) is passed explicitly to this stage; without it,
primary-twin introduction targets the full combined EBSD twin fraction
independently, and secondary-twin introduction is never triggered.
`TWIN_TVF_TOLERANCE` specifies the permitted overshoot beyond the target
volume fraction before introduction halts; `TWIN_PROB_SEC_OUTWARD`/
`TWIN_PROB_SEC_INWARD` specify the 2a (outward, host-interface) versus 2b
(inward, nested within the primary) split for secondary-twin
nucleation.

In [ ]:
TWIN_N_LAMELLAE_PER_HOST = 10
TWIN_TVF_TOLERANCE = 0.05          # allowed overshoot past each stage's VF target before stopping
TWIN_TVF_2D_TO_3D_SCALE = 1.15     # fallback only -- unused once vf_targets (staged) is provided
TWIN_THICK_SCALE_FACTOR = 0.80     # scales the EBSD-measured thickness distribution used for lamellae
TWIN_MAX_VF_PER_HOST = 0.50        # cap on twin volume as a fraction of its own host grain's volume
TWIN_PROB_SEC_OUTWARD = 0.60       # secondary-twin nucleation: 2a (outward, at host interface)
TWIN_PROB_SEC_INWARD = 0.40        # secondary-twin nucleation: 2b (inward, nested in primary)
TWIN_RNG_SEED = 123

tg, tvf = steps_twin_generation.generate_twins(
    base, rg, parent_info, assigner, twin_thickness, vf_targets=vf_targets, csl_label=CSL_LABEL,
    n_lamellae_per_host=TWIN_N_LAMELLAE_PER_HOST, nucleation_site='gb_centroid', meshing_route='conformal',
    tvf_tolerance=TWIN_TVF_TOLERANCE, tvf_2d_to_3d_scale=TWIN_TVF_2D_TO_3D_SCALE,
    thick_scale_factor=TWIN_THICK_SCALE_FACTOR, max_vf_per_host=TWIN_MAX_VF_PER_HOST,
    prob_sec_outward=TWIN_PROB_SEC_OUTWARD, prob_sec_inward=TWIN_PROB_SEC_INWARD,
    rng_seed=TWIN_RNG_SEED)
tg.summary()

In [ ]:
cleaner, n_passes_run = steps_twin_generation.clean_structure(tg, n_passes=5)
len(cleaner.twin_role_clean), n_passes_run

## Part M: Post-Twin Validation


Validates the misorientation distribution of the cleaned, twinned
structure against the full EBSD MDF (twins now present, in contrast to
Pre-Twin Validation), per axis, and compares the full three-dimensional
MDF curve directly against the EBSD reference curve computed in
Part D.

In [ ]:
post_validator, quat_3d_clean = steps_post_twin_validation.validate_crystallographic_representativeness(
    cleaner, mdf, n_slices=(10, 10, 10), test_axes=('x', 'y', 'z'),
    p_percentage=60.0, wasserstein_threshold=5.0)
post_validator.overall_accepted

In [ ]:
mc_mdf_post = steps_post_twin_validation.compare_full_3d_mdf(cleaner, quat_3d_clean)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mdf['hist_bin_centers'], mdf['hist_density'], label='EBSD')
ax.plot(mc_mdf_post['hist_bin_centers'], mc_mdf_post['hist_density'], label='SGC (post-twin)')
ax.set_xlabel('Misorientation angle (deg)'); ax.set_ylabel('Density'); ax.legend()
ax.set_title('Full 3D MDF: EBSD vs. Synthetic Grain Structure')
report.add_image(fig, title='Post-Twin Full 3D MDF', section='Post-Twin Validation')
plt.show()

## Part N: Distribution Viewer (Omitted)

The EBSD-versus-synthetic-structure comparison overlay, omitted in this
basic walkthrough for brevity. See
`twinned_fcc_int0.ipynb`/`twinned_fcc_adv0.ipynb` for the complete
comparison.

## Part O: Subsetting (Omitted)

Optional cuboidal sub-volume extraction, omitted in this basic
walkthrough; Visualization and Export operates directly on the full
cleaned structure. See `twinned_fcc_int0.ipynb`/`twinned_fcc_adv0.ipynb`
to enable this stage.

## Part P: Visualization and Export

Renders the finished three-dimensional structure interactively, writes
the cleaned structure's raw arrays (`.npy`/`.pkl`), and exports a full
Abaqus voxel-conformal mesh (`.inp` files) with element sets by role,
family, and variant, and node sets on each domain face.

In [ ]:
# Interactive PyVista render -- opens a separate window (or renders inline
# with a Jupyter PyVista backend enabled). Skip this cell in a headless run.
steps_visualization_export.render_3d_structure(cleaner)

In [ ]:
raw_result = steps_visualization_export.export_raw(cleaner)
raw_result

In [ ]:
index = steps_visualization_export.build_abaqus_index(cleaner)

In [ ]:
abq_result = steps_visualization_export.export_abaqus_mesh(
    cleaner, tg, index, base_filename='twinned_fcc_mesh', voxel_size_um=1.0,
    element_type='C3D8', material_format='bunge_euler', n_depvar=100,
    material_level='feature')
abq_result

In [ ]:
abq_summary = {k: v for k, v in abq_result.items() if k != 'files'}
report.add_table([abq_summary], title='Abaqus Mesh Export', section='Visualization & Export')

## Summary: Achieved versus Target Total Twin Volume Fraction

Reports the fraction of the final, cleaned structure's volume occupied by
twin material (primary and secondary combined), relative to the full
EBSD-partitioned target (Stage-1 + Secondary-2a + Secondary-2b from
`vf_targets`, Part D). The comparison is evaluated on `cleaner`, not `tg`,
since cleaning (spike removal, lobe splitting) alters voxel counts
following twin introduction. The value
`tg.summary()['tvf_achieved_pct_of_target']`, reported above, does not
represent this quantity: it compares the pre-cleaning, all-twins-combined
volume against `tvf_target_3d`, the Stage-1-only target used as the
stopping threshold by `introduce_primary_twins`, rather than the combined
total, and consequently reads higher than the true achieved fraction once
secondary twins are included.

In [ ]:
twin_gids = {g for g, role in cleaner.twin_role_clean.items()
             if role in ('primary_twin', 'secondary_twin')}
twin_vox = int(np.isin(cleaner.lgi_clean, list(twin_gids)).sum())
total_vox = int(np.sum(cleaner.lgi_clean > 0))
achieved_total_vf = twin_vox / total_vox if total_vox > 0 else 0.0

target_total_vf = (vf_targets['tvf_stage1'] + vf_targets['tvf_secondary_2a']
                    + vf_targets['tvf_secondary_2b'])
pct_of_target = 100 * achieved_total_vf / target_total_vf if target_total_vf > 0 else 0.0

print(f'Target total twin VF (EBSD-partitioned, 3D) : {target_total_vf:.4f}')
print(f'Achieved total twin VF (cleaned structure)   : {achieved_total_vf:.4f}')
print(f'Achieved / Target                            : {pct_of_target:.1f}%')

report.add_table(
    [{'target_total_vf': target_total_vf, 'achieved_total_vf': achieved_total_vf,
      'pct_of_target': pct_of_target}],
    title='Achieved vs. Target Total Twin Volume Fraction', section='Twin Generation')

## Finish: Report Generation

Renders the accumulated `add_table`/`add_image` entries into a single
`report.html` file.

In [ ]:
report_path = steps_visualization_export.finish_report(report)
print('Report written to:', report_path)